# Phase 4C: paired raw/SRO resume to epoch 2
One notebook URL; each saved version advances both branches by one epoch. Inputs must pin the prior successful outputs, original train data, and SRO dataset.


In [1]:
from pathlib import Path
import subprocess
import sys
branch = 'raw'
expected_prior_epoch = 1
source = '"""One resumable phase-4C Kaggle training segment; never reads MuSiQue dev."""\n\nimport csv\nimport hashlib\nimport inspect\nimport json\nimport math\nimport random\nimport re\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport transformers\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n                          DataCollatorWithPadding, EarlyStoppingCallback,\n                          Trainer, TrainerCallback, TrainingArguments)\n\n\nBRANCH = "raw"  # Generate a separate notebook with "sro" for the paired branch.\nMODE = "train"  # "profile" or "train"; profile does not save a checkpoint.\nTARGET_EPOCH = 2  # Train through this epoch, then stop and save for resume.\nPRIOR_ROOT = "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-raw-runtime-profile/phase4c_raw"  # Mounted previous run output directory, or None for epoch 1.\nRUN_ID = "phase4c_raw_epoch2_20261122_01"\n\nBASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data")\nSRO = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121")\nOUT = Path("/kaggle/working") / f"phase4c_{BRANCH}"\nMODEL = "microsoft/deberta-v3-large"\nREVISION = "64a8c8eab3e352a784c658aef62be1662607476f"\nSEED = 42\nSPLIT_HASH = "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0"\nRAW_HASH = "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D"\nSRO_HASH = "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8"\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest().upper()\n\n\ndef rows(path):\n    with path.open(encoding="utf-8-sig", newline="") as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != ["text", "label"]:\n            raise ValueError(f"invalid CSV header: {path}")\n        data = list(reader)\n    if len(data) != 46610:\n        raise ValueError(f"wrong train rows: {path}")\n    return data\n\n\ndef split_key(text):\n    prefix, separator = "Sub-question: ", "\\n\\nPassages:\\n"\n    if not text.startswith(prefix) or separator not in text:\n        raise ValueError("invalid text contract")\n    return re.sub(r"\\s+", " ", text[len(prefix):].split(separator, 1)[0]).strip().casefold()\n\n\ndef split_indices(raw, sro):\n    indices = {"train": [], "validation": []}\n    groups = {}\n    for i, (a, b) in enumerate(zip(raw, sro)):\n        if a["label"] not in {"0", "1"} or a["label"] != b["label"]:\n            raise ValueError(f"label mismatch at {i}")\n        sep = "\\n\\nPassages:\\n"\n        if a["text"].split(sep, 1)[1] != b["text"].split(sep, 1)[1]:\n            raise ValueError(f"passage mismatch at {i}")\n        key = split_key(a["text"])\n        digest = hashlib.sha256(f"{SEED}\\n{key}".encode()).digest()\n        fold = "validation" if int.from_bytes(digest[:8], "big") < int(0.1 * 2**64) else "train"\n        if key in groups and groups[key] != fold:\n            raise ValueError("group leakage")\n        groups[key] = fold\n        indices[fold].append(i)\n    # The frozen indices.json was written on Windows with a CRLF final newline.\n    payload = (json.dumps(indices, separators=(",", ":")) + "\\r\\n").encode()\n    if hashlib.sha256(payload).hexdigest().upper() != SPLIT_HASH:\n        raise ValueError("frozen validation split hash mismatch")\n    if len(indices["train"]) != 42211 or len(indices["validation"]) != 4399:\n        raise ValueError("frozen validation counts mismatch")\n    return indices\n\n\nclass Encoded(torch.utils.data.Dataset):\n    def __init__(self, records, tokenizer):\n        texts = [r["text"] for r in records]\n        self.labels = [int(r["label"]) for r in records]\n        self.encodings = tokenizer(texts, max_length=512, truncation=True, padding=False)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, i):\n        return {**{k: v[i] for k, v in self.encodings.items()}, "labels": self.labels[i]}\n\n\ndef metrics(prediction):\n    probs = torch.softmax(torch.tensor(prediction.predictions), dim=-1)[:, 1].numpy()\n    y = np.asarray(prediction.label_ids)\n    p = (probs >= 0.5).astype(int)\n    tp = int(((y == 1) & (p == 1)).sum())\n    fp = int(((y == 0) & (p == 1)).sum())\n    fn = int(((y == 1) & (p == 0)).sum())\n    tn = int(((y == 0) & (p == 0)).sum())\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    return {"precision": precision, "recall": recall,\n            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,\n            "accuracy": (tp + tn) / len(y), "flag_rate": (tn + fn) / len(y)}\n\n\nclass SegmentStop(TrainerCallback):\n    def on_epoch_end(self, args, state, control, **kwargs):\n        if state.epoch >= TARGET_EPOCH:\n            control.should_training_stop = True\n        return control\n\n\ndef main():\n    if BRANCH not in {"raw", "sro"} or MODE not in {"profile", "train"}:\n        raise ValueError("invalid branch or mode")\n    if MODE == "train" and not 1 <= TARGET_EPOCH <= 10:\n        raise ValueError("target epoch out of range")\n    raw_path, sro_path = BASE / "deberta_fg_train.csv", SRO / "train.csv"\n    if sha(raw_path) != RAW_HASH or sha(sro_path) != SRO_HASH:\n        raise ValueError("source train CSV hash mismatch")\n    raw, sro = rows(raw_path), rows(sro_path)\n    indices = split_indices(raw, sro)\n    branch_rows = raw if BRANCH == "raw" else sro\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError("expected Kaggle T4 x2")\n    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)\n    train_records = [branch_rows[i] for i in indices["train"]]\n    validation_records = [branch_rows[i] for i in indices["validation"]]\n    train_ds = Encoded(train_records[:800] if MODE == "profile" else train_records, tokenizer)\n    val_ds = Encoded(validation_records, tokenizer) if MODE == "train" else None\n    model = AutoModelForSequenceClassification.from_pretrained(\n        MODEL, revision=REVISION, num_labels=2,\n        dtype=torch.float32,\n        id2label={0: "NOT-ANSWERABLE", 1: "ANSWERABLE"},\n        label2id={"NOT-ANSWERABLE": 0, "ANSWERABLE": 1})\n    # FP16 AMP keeps trainable weights/grads in FP32; otherwise GradScaler\n    # rejects FP16 gradients when unscaling on Kaggle\'s Transformers 5 image.\n    trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}\n    if trainable_dtypes != {torch.float32}:\n        raise RuntimeError(f"expected FP32 trainable weights for FP16 AMP: {trainable_dtypes}")\n    if OUT.exists():\n        raise FileExistsError(OUT)\n    prior_progress = None\n    if PRIOR_ROOT is None:\n        OUT.mkdir(parents=True)\n        resume = None\n    else:\n        prior = Path(PRIOR_ROOT)\n        prior_progress = json.loads((prior / "progress.json").read_text())\n        if prior_progress["branch"] != BRANCH or prior_progress["split_sha256"] != SPLIT_HASH:\n            raise ValueError("prior run branch or split mismatch")\n        shutil.copytree(prior / "checkpoints", OUT / "checkpoints")\n        resume = max((OUT / "checkpoints").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))\n    kwargs = dict(output_dir=str(OUT / "checkpoints"), num_train_epochs=10,\n                  per_device_train_batch_size=4, per_device_eval_batch_size=4,\n                  gradient_accumulation_steps=2, learning_rate=2e-5, weight_decay=0.01,\n                  lr_scheduler_type="linear", fp16=True,\n                  gradient_checkpointing=True,\n                  gradient_checkpointing_kwargs={"use_reentrant": False},\n                  optim="adamw_torch", logging_steps=10,\n                  disable_tqdm=True, report_to="none", seed=SEED, data_seed=SEED,\n                  dataloader_num_workers=2, save_strategy="epoch" if MODE == "train" else "no",\n                  load_best_model_at_end=MODE == "train", metric_for_best_model="f1" if MODE == "train" else None,\n                  greater_is_better=True, save_total_limit=2,\n                  max_steps=50 if MODE == "profile" else -1)\n    supported = inspect.signature(TrainingArguments).parameters\n    # Transformers 5.20 moved ratio warmup into warmup_steps and renamed\n    # group_by_length to train_sampling_strategy. Both express the same setup.\n    if "warmup_ratio" in supported:\n        kwargs["warmup_ratio"] = 0.1\n    else:\n        kwargs["warmup_steps"] = 0.1\n    if "group_by_length" in supported:\n        kwargs["group_by_length"] = True\n    else:\n        kwargs["train_sampling_strategy"] = "group_by_length"\n    strategy_name = "eval_strategy" if "eval_strategy" in supported else "evaluation_strategy"\n    kwargs[strategy_name] = "epoch" if MODE == "train" else "no"\n    unsupported = set(kwargs) - set(supported)\n    if unsupported:\n        raise TypeError(f"unsupported TrainingArguments fields: {sorted(unsupported)}")\n    args = TrainingArguments(**kwargs)\n    callbacks = [SegmentStop()]\n    early_stopping = None\n    if MODE == "train":\n        early_stopping = EarlyStoppingCallback(early_stopping_patience=3)\n        if prior_progress:\n            early_stopping.early_stopping_patience_counter = prior_progress["early_stopping_patience_counter"]\n        callbacks.append(early_stopping)\n    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,\n                      compute_metrics=metrics if MODE == "train" else None,\n                      data_collator=DataCollatorWithPadding(tokenizer, padding="longest", pad_to_multiple_of=8),\n                      callbacks=callbacks)\n    started = time.monotonic()\n    result = trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    elapsed = time.monotonic() - started\n    if MODE == "profile":\n        projected = elapsed / trainer.state.global_step * math.ceil(42211 / 16)\n        report = {"status": "profile_only", "branch": BRANCH, "steps": trainer.state.global_step,\n                  "elapsed_seconds": elapsed, "projected_one_epoch_training_seconds": projected,\n                  "under_9_hour_guard": projected < 9 * 3600,\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        (OUT / "profile.json").write_text(json.dumps(report, indent=2) + "\\n")\n    else:\n        state = trainer.state\n        report = {"status": "checkpoint_saved", "run_id": RUN_ID, "branch": BRANCH,\n                  "target_epoch": TARGET_EPOCH, "completed_epoch": state.epoch,\n                  "global_step": state.global_step, "best_validation_f1": state.best_metric,\n                  "best_checkpoint": state.best_model_checkpoint,\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "raw_sha256": RAW_HASH, "sro_sha256": SRO_HASH,\n                  "train_rows": 42211, "validation_rows": 4399,\n                  "early_stopping_patience_counter": early_stopping.early_stopping_patience_counter,\n                  "training_seconds": elapsed, "train_metrics": result.metrics,\n                  "history": state.log_history,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        (OUT / "progress.json").write_text(json.dumps(report, indent=2, default=str) + "\\n")\n    print(json.dumps(report, indent=2, default=str), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
prior = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-raw-runtime-profile/phase4c_raw")
if not (prior / "progress.json").is_file():
    raise FileNotFoundError(f"Attach the exact prior notebook output: {prior}")
import json
progress = json.loads((prior / "progress.json").read_text())
if progress["branch"] != branch or progress["completed_epoch"] != expected_prior_epoch:
    raise ValueError(f"Wrong prior run for {branch}: {progress}")
if progress["global_step"] != 2639 * expected_prior_epoch:
    raise ValueError("Prior run did not complete the expected steps")
if progress["split_sha256"] != "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0":
    raise ValueError("Frozen split mismatch")
path = Path("/kaggle/working/_phase4c_resume_raw.py")
path.write_text(source, encoding="utf-8")
subprocess.run([sys.executable, str(path)], check=True)


Loading weights: 100%|██████████| 390/390 [00:00<00:00, 724.63it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.weight                     | MISSING    

{'loss': '0.81', 'grad_norm': '16.59', 'learning_rate': '2e-05', 'epoch': '1'}
{'loss': '1.613', 'grad_norm': '25.26', 'learning_rate': '1.999e-05', 'epoch': '1.004'}
{'loss': '1.438', 'grad_norm': '17.18', 'learning_rate': '1.998e-05', 'epoch': '1.008'}
{'loss': '1.123', 'grad_norm': '16.15', 'learning_rate': '1.997e-05', 'epoch': '1.012'}
{'loss': '1.458', 'grad_norm': '32.49', 'learning_rate': '1.997e-05', 'epoch': '1.016'}
{'loss': '1.274', 'grad_norm': '24.47', 'learning_rate': '1.996e-05', 'epoch': '1.019'}
{'loss': '1.008', 'grad_norm': '23.33', 'learning_rate': '1.995e-05', 'epoch': '1.023'}
{'loss': '1.301', 'grad_norm': '36.22', 'learning_rate': '1.994e-05', 'epoch': '1.027'}
{'loss': '1.444', 'grad_norm': '23.2', 'learning_rate': '1.993e-05', 'epoch': '1.031'}
{'loss': '1.576', 'grad_norm': '29.81', 'learning_rate': '1.992e-05', 'epoch': '1.034'}
{'loss': '0.8514', 'grad_norm': '17.68', 'learning_rate': '1.992e-05', 'epoch': '1.038'}
{'loss': '1.249', 'grad_norm': '13.93', '

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.72s/it]


{'train_runtime': '1.143e+04', 'train_samples_per_second': '3.695', 'train_steps_per_second': '0.231', 'train_loss': '1.259', 'epoch': '2'}
{
  "status": "checkpoint_saved",
  "run_id": "phase4c_raw_epoch2_20261122_01",
  "branch": "raw",
  "target_epoch": 2,
  "completed_epoch": 2.0,
  "global_step": 5278,
  "best_validation_f1": 0.8087039693926351,
  "best_checkpoint": "/kaggle/working/phase4c_raw/checkpoints/checkpoint-2639",
  "split_sha256": "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0",
  "model_revision": "64a8c8eab3e352a784c658aef62be1662607476f",
  "raw_sha256": "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D",
  "sro_sha256": "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8",
  "train_rows": 42211,
  "validation_rows": 4399,
  "early_stopping_patience_counter": 1,
  "training_seconds": 11435.188616116,
  "train_metrics": {
    "train_runtime": 11425.1423,
    "train_samples_per_second": 3.695,
    "train_steps_per_second

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/_phase4c_resume_raw.py'], returncode=0)

In [2]:
from pathlib import Path
import subprocess
import sys
branch = 'sro'
expected_prior_epoch = 1
source = '"""One resumable phase-4C Kaggle training segment; never reads MuSiQue dev."""\n\nimport csv\nimport hashlib\nimport inspect\nimport json\nimport math\nimport random\nimport re\nimport shutil\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport transformers\nfrom transformers import (AutoModelForSequenceClassification, AutoTokenizer,\n                          DataCollatorWithPadding, EarlyStoppingCallback,\n                          Trainer, TrainerCallback, TrainingArguments)\n\n\nBRANCH = "sro"  # Generate a separate notebook with "sro" for the paired branch.\nMODE = "train"  # "profile" or "train"; profile does not save a checkpoint.\nTARGET_EPOCH = 2  # Train through this epoch, then stop and save for resume.\nPRIOR_ROOT = "/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-sro-runtime-profile/phase4c_sro"  # Mounted previous run output directory, or None for epoch 1.\nRUN_ID = "phase4c_sro_epoch2_20261122_01"\n\nBASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data")\nSRO = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121")\nOUT = Path("/kaggle/working") / f"phase4c_{BRANCH}"\nMODEL = "microsoft/deberta-v3-large"\nREVISION = "64a8c8eab3e352a784c658aef62be1662607476f"\nSEED = 42\nSPLIT_HASH = "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0"\nRAW_HASH = "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D"\nSRO_HASH = "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8"\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest().upper()\n\n\ndef rows(path):\n    with path.open(encoding="utf-8-sig", newline="") as f:\n        reader = csv.DictReader(f)\n        if reader.fieldnames != ["text", "label"]:\n            raise ValueError(f"invalid CSV header: {path}")\n        data = list(reader)\n    if len(data) != 46610:\n        raise ValueError(f"wrong train rows: {path}")\n    return data\n\n\ndef split_key(text):\n    prefix, separator = "Sub-question: ", "\\n\\nPassages:\\n"\n    if not text.startswith(prefix) or separator not in text:\n        raise ValueError("invalid text contract")\n    return re.sub(r"\\s+", " ", text[len(prefix):].split(separator, 1)[0]).strip().casefold()\n\n\ndef split_indices(raw, sro):\n    indices = {"train": [], "validation": []}\n    groups = {}\n    for i, (a, b) in enumerate(zip(raw, sro)):\n        if a["label"] not in {"0", "1"} or a["label"] != b["label"]:\n            raise ValueError(f"label mismatch at {i}")\n        sep = "\\n\\nPassages:\\n"\n        if a["text"].split(sep, 1)[1] != b["text"].split(sep, 1)[1]:\n            raise ValueError(f"passage mismatch at {i}")\n        key = split_key(a["text"])\n        digest = hashlib.sha256(f"{SEED}\\n{key}".encode()).digest()\n        fold = "validation" if int.from_bytes(digest[:8], "big") < int(0.1 * 2**64) else "train"\n        if key in groups and groups[key] != fold:\n            raise ValueError("group leakage")\n        groups[key] = fold\n        indices[fold].append(i)\n    # The frozen indices.json was written on Windows with a CRLF final newline.\n    payload = (json.dumps(indices, separators=(",", ":")) + "\\r\\n").encode()\n    if hashlib.sha256(payload).hexdigest().upper() != SPLIT_HASH:\n        raise ValueError("frozen validation split hash mismatch")\n    if len(indices["train"]) != 42211 or len(indices["validation"]) != 4399:\n        raise ValueError("frozen validation counts mismatch")\n    return indices\n\n\nclass Encoded(torch.utils.data.Dataset):\n    def __init__(self, records, tokenizer):\n        texts = [r["text"] for r in records]\n        self.labels = [int(r["label"]) for r in records]\n        self.encodings = tokenizer(texts, max_length=512, truncation=True, padding=False)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, i):\n        return {**{k: v[i] for k, v in self.encodings.items()}, "labels": self.labels[i]}\n\n\ndef metrics(prediction):\n    probs = torch.softmax(torch.tensor(prediction.predictions), dim=-1)[:, 1].numpy()\n    y = np.asarray(prediction.label_ids)\n    p = (probs >= 0.5).astype(int)\n    tp = int(((y == 1) & (p == 1)).sum())\n    fp = int(((y == 0) & (p == 1)).sum())\n    fn = int(((y == 1) & (p == 0)).sum())\n    tn = int(((y == 0) & (p == 0)).sum())\n    precision = tp / (tp + fp) if tp + fp else 0.0\n    recall = tp / (tp + fn) if tp + fn else 0.0\n    return {"precision": precision, "recall": recall,\n            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,\n            "accuracy": (tp + tn) / len(y), "flag_rate": (tn + fn) / len(y)}\n\n\nclass SegmentStop(TrainerCallback):\n    def on_epoch_end(self, args, state, control, **kwargs):\n        if state.epoch >= TARGET_EPOCH:\n            control.should_training_stop = True\n        return control\n\n\ndef main():\n    if BRANCH not in {"raw", "sro"} or MODE not in {"profile", "train"}:\n        raise ValueError("invalid branch or mode")\n    if MODE == "train" and not 1 <= TARGET_EPOCH <= 10:\n        raise ValueError("target epoch out of range")\n    raw_path, sro_path = BASE / "deberta_fg_train.csv", SRO / "train.csv"\n    if sha(raw_path) != RAW_HASH or sha(sro_path) != SRO_HASH:\n        raise ValueError("source train CSV hash mismatch")\n    raw, sro = rows(raw_path), rows(sro_path)\n    indices = split_indices(raw, sro)\n    branch_rows = raw if BRANCH == "raw" else sro\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    if torch.cuda.device_count() != 2:\n        raise RuntimeError("expected Kaggle T4 x2")\n    tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)\n    train_records = [branch_rows[i] for i in indices["train"]]\n    validation_records = [branch_rows[i] for i in indices["validation"]]\n    train_ds = Encoded(train_records[:800] if MODE == "profile" else train_records, tokenizer)\n    val_ds = Encoded(validation_records, tokenizer) if MODE == "train" else None\n    model = AutoModelForSequenceClassification.from_pretrained(\n        MODEL, revision=REVISION, num_labels=2,\n        dtype=torch.float32,\n        id2label={0: "NOT-ANSWERABLE", 1: "ANSWERABLE"},\n        label2id={"NOT-ANSWERABLE": 0, "ANSWERABLE": 1})\n    # FP16 AMP keeps trainable weights/grads in FP32; otherwise GradScaler\n    # rejects FP16 gradients when unscaling on Kaggle\'s Transformers 5 image.\n    trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}\n    if trainable_dtypes != {torch.float32}:\n        raise RuntimeError(f"expected FP32 trainable weights for FP16 AMP: {trainable_dtypes}")\n    if OUT.exists():\n        raise FileExistsError(OUT)\n    prior_progress = None\n    if PRIOR_ROOT is None:\n        OUT.mkdir(parents=True)\n        resume = None\n    else:\n        prior = Path(PRIOR_ROOT)\n        prior_progress = json.loads((prior / "progress.json").read_text())\n        if prior_progress["branch"] != BRANCH or prior_progress["split_sha256"] != SPLIT_HASH:\n            raise ValueError("prior run branch or split mismatch")\n        shutil.copytree(prior / "checkpoints", OUT / "checkpoints")\n        resume = max((OUT / "checkpoints").glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))\n    kwargs = dict(output_dir=str(OUT / "checkpoints"), num_train_epochs=10,\n                  per_device_train_batch_size=4, per_device_eval_batch_size=4,\n                  gradient_accumulation_steps=2, learning_rate=2e-5, weight_decay=0.01,\n                  lr_scheduler_type="linear", fp16=True,\n                  gradient_checkpointing=True,\n                  gradient_checkpointing_kwargs={"use_reentrant": False},\n                  optim="adamw_torch", logging_steps=10,\n                  disable_tqdm=True, report_to="none", seed=SEED, data_seed=SEED,\n                  dataloader_num_workers=2, save_strategy="epoch" if MODE == "train" else "no",\n                  load_best_model_at_end=MODE == "train", metric_for_best_model="f1" if MODE == "train" else None,\n                  greater_is_better=True, save_total_limit=2,\n                  max_steps=50 if MODE == "profile" else -1)\n    supported = inspect.signature(TrainingArguments).parameters\n    # Transformers 5.20 moved ratio warmup into warmup_steps and renamed\n    # group_by_length to train_sampling_strategy. Both express the same setup.\n    if "warmup_ratio" in supported:\n        kwargs["warmup_ratio"] = 0.1\n    else:\n        kwargs["warmup_steps"] = 0.1\n    if "group_by_length" in supported:\n        kwargs["group_by_length"] = True\n    else:\n        kwargs["train_sampling_strategy"] = "group_by_length"\n    strategy_name = "eval_strategy" if "eval_strategy" in supported else "evaluation_strategy"\n    kwargs[strategy_name] = "epoch" if MODE == "train" else "no"\n    unsupported = set(kwargs) - set(supported)\n    if unsupported:\n        raise TypeError(f"unsupported TrainingArguments fields: {sorted(unsupported)}")\n    args = TrainingArguments(**kwargs)\n    callbacks = [SegmentStop()]\n    early_stopping = None\n    if MODE == "train":\n        early_stopping = EarlyStoppingCallback(early_stopping_patience=3)\n        if prior_progress:\n            early_stopping.early_stopping_patience_counter = prior_progress["early_stopping_patience_counter"]\n        callbacks.append(early_stopping)\n    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,\n                      compute_metrics=metrics if MODE == "train" else None,\n                      data_collator=DataCollatorWithPadding(tokenizer, padding="longest", pad_to_multiple_of=8),\n                      callbacks=callbacks)\n    started = time.monotonic()\n    result = trainer.train(resume_from_checkpoint=str(resume) if resume else None)\n    elapsed = time.monotonic() - started\n    if MODE == "profile":\n        projected = elapsed / trainer.state.global_step * math.ceil(42211 / 16)\n        report = {"status": "profile_only", "branch": BRANCH, "steps": trainer.state.global_step,\n                  "elapsed_seconds": elapsed, "projected_one_epoch_training_seconds": projected,\n                  "under_9_hour_guard": projected < 9 * 3600,\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        (OUT / "profile.json").write_text(json.dumps(report, indent=2) + "\\n")\n    else:\n        state = trainer.state\n        report = {"status": "checkpoint_saved", "run_id": RUN_ID, "branch": BRANCH,\n                  "target_epoch": TARGET_EPOCH, "completed_epoch": state.epoch,\n                  "global_step": state.global_step, "best_validation_f1": state.best_metric,\n                  "best_checkpoint": state.best_model_checkpoint,\n                  "split_sha256": SPLIT_HASH, "model_revision": REVISION,\n                  "raw_sha256": RAW_HASH, "sro_sha256": SRO_HASH,\n                  "train_rows": 42211, "validation_rows": 4399,\n                  "early_stopping_patience_counter": early_stopping.early_stopping_patience_counter,\n                  "training_seconds": elapsed, "train_metrics": result.metrics,\n                  "history": state.log_history,\n                  "torch": torch.__version__, "transformers": transformers.__version__}\n        (OUT / "progress.json").write_text(json.dumps(report, indent=2, default=str) + "\\n")\n    print(json.dumps(report, indent=2, default=str), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
prior = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-sro-runtime-profile/phase4c_sro")
if not (prior / "progress.json").is_file():
    raise FileNotFoundError(f"Attach the exact prior notebook output: {prior}")
import json
progress = json.loads((prior / "progress.json").read_text())
if progress["branch"] != branch or progress["completed_epoch"] != expected_prior_epoch:
    raise ValueError(f"Wrong prior run for {branch}: {progress}")
if progress["global_step"] != 2639 * expected_prior_epoch:
    raise ValueError("Prior run did not complete the expected steps")
if progress["split_sha256"] != "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0":
    raise ValueError("Frozen split mismatch")
path = Path("/kaggle/working/_phase4c_resume_sro.py")
path.write_text(source, encoding="utf-8")
subprocess.run([sys.executable, str(path)], check=True)


Loading weights: 100%|██████████| 390/390 [00:00<00:00, 745.67it/s]
[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    

{'loss': '0.9419', 'grad_norm': '20.97', 'learning_rate': '2e-05', 'epoch': '1'}
{'loss': '2.007', 'grad_norm': '44.19', 'learning_rate': '1.999e-05', 'epoch': '1.004'}
{'loss': '1.812', 'grad_norm': '15.29', 'learning_rate': '1.998e-05', 'epoch': '1.008'}
{'loss': '1.696', 'grad_norm': '36.87', 'learning_rate': '1.997e-05', 'epoch': '1.012'}
{'loss': '1.867', 'grad_norm': '87.65', 'learning_rate': '1.997e-05', 'epoch': '1.016'}
{'loss': '1.63', 'grad_norm': '26.31', 'learning_rate': '1.996e-05', 'epoch': '1.019'}
{'loss': '1.748', 'grad_norm': '12.67', 'learning_rate': '1.995e-05', 'epoch': '1.023'}
{'loss': '1.73', 'grad_norm': '19.95', 'learning_rate': '1.994e-05', 'epoch': '1.027'}
{'loss': '1.899', 'grad_norm': '8.383', 'learning_rate': '1.993e-05', 'epoch': '1.031'}
{'loss': '2.12', 'grad_norm': '22.61', 'learning_rate': '1.992e-05', 'epoch': '1.034'}
{'loss': '1.645', 'grad_norm': '37.04', 'learning_rate': '1.992e-05', 'epoch': '1.038'}
{'loss': '1.644', 'grad_norm': '13.84', 'l

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]


{'train_runtime': '1.408e+04', 'train_samples_per_second': '2.998', 'train_steps_per_second': '0.187', 'train_loss': '1.711', 'epoch': '2'}
{
  "status": "checkpoint_saved",
  "run_id": "phase4c_sro_epoch2_20261122_01",
  "branch": "sro",
  "target_epoch": 2,
  "completed_epoch": 2.0,
  "global_step": 5278,
  "best_validation_f1": 0.7822410147991543,
  "best_checkpoint": "/kaggle/working/phase4c_sro/checkpoints/checkpoint-5278",
  "split_sha256": "7C9435008AD7723060EB65FFBEE59722AABD31FC028EDA1E765248414ACFE6F0",
  "model_revision": "64a8c8eab3e352a784c658aef62be1662607476f",
  "raw_sha256": "7832387E08D082E4E2CA8B4161AEFEA2CC56450E003E14E312C3BC5F31C0557D",
  "sro_sha256": "A94ECA9FA46D41D2F4DE706E03842A30C438F3792A88F9425CF9594EECF85AA8",
  "train_rows": 42211,
  "validation_rows": 4399,
  "early_stopping_patience_counter": 0,
  "training_seconds": 14084.853381526,
  "train_metrics": {
    "train_runtime": 14079.9926,
    "train_samples_per_second": 2.998,
    "train_steps_per_second

CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/_phase4c_resume_sro.py'], returncode=0)